# Run Qwen2.5-1.5B representation geometry in Google Colab

**Two files:** upload this notebook using Colab's **File → Upload notebook**, then upload `intrinsic-dimension-transformers-v2-colab.zip` when the first code cell asks. Choose **Runtime → Change runtime type → GPU** before running cells top to bottom. A T4 is suitable when available; the notebook checks the actual GPU.

This uses the pretrained model; **no training or fine-tuning**. Version 2 uses 640 prompts, 20 named task families per category and 10 held-out families per category. Actual model-token length and family coverage checks must pass before loading weights. The project compares final input-token representation clouds using participation ratio, covariance effective rank, PCA spectra and TWO-NN, without assuming a task ordering.

**Figure quality:** all scientific figures export at **600 DPI as PNG**, plus **vector PDF and SVG** with larger fonts. Use PDF/SVG for papers or zooming; retain source CSVs. All synthetic validation is clearly labeled and stored separately from real output.

Run the synthetic checks before model extraction. Clicking the extraction cell downloads model weights and runs every hidden-state layer. Drive storage is enabled for this run at your request; authorize the Drive prompt when you run that cell. Set `SAVE_TO_DRIVE=False` if you later choose temporary storage. Without Drive, download results before ending the session.

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, importlib.metadata, hashlib, json
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# User settings: choose Drive storage explicitly if wanted.
SAVE_TO_DRIVE = IN_COLAB  # Drive storage requested for this corrected run.
BATCH_SIZE = 1                    # Conservative starting point for FP32 on T4.
FIGURE_DPI = 600                  # PNG resolution; PDF/SVG are vector.
MODEL_REVISION = "8faed761d45a263340a0528343f099c05c9a4323"  # Same as the pilot.
RUN_REAL_MODEL = IN_COLAB         # True in Colab; local notebook validation skips downloads.
INSTALL_DEPENDENCIES = IN_COLAB   # Keeps preinstalled compatible Colab PyTorch.

PROJECT = Path("/content/intrinsic-dimension-transformers") if IN_COLAB else Path.cwd()
if not IN_COLAB and not (PROJECT / "pyproject.toml").exists():
    PROJECT = PROJECT.parent
EXPECTED_SOURCE = "4e81f5f00fa5e7027d6389e464b5f4227cea69955ef0818c393b655bd5efd09e"
def source_matches():
    package = PROJECT / "src/intrinsic_dim"
    if not (PROJECT / "data/design_manifest_v2.json").exists():
        return False
    digests = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(package.glob("*.py"))}
    return hashlib.sha256(json.dumps(digests, sort_keys=True).encode()).hexdigest() == EXPECTED_SOURCE
source_updated = False
if not source_matches():
    assert IN_COLAB, "Run locally from the project or notebooks folder"
    print("Upload intrinsic-dimension-transformers-v2-colab.zip now")
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one project ZIP"
    zip_path = Path(next(iter(uploaded)))
    assert zipfile.is_zipfile(zip_path), "Choose the project .zip, not the notebook"
    target = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            resolved = (target / member.filename).resolve()
            assert target in resolved.parents, "Unsafe ZIP path"
            assert (member.external_attr >> 16) & 0o170000 != 0o120000, "ZIP symlinks are unsupported"
        archive.extractall(target)
    source_updated = True
assert source_matches(), "Upload the current v2 ZIP; older source cannot pass this check"
if source_updated and any(name == "intrinsic_dim" or name.startswith("intrinsic_dim.") for name in sys.modules):
    raise RuntimeError("Source updated but old Python modules are loaded. Restart session and rerun setup; cached files and weights can be reused.")
assert (PROJECT / "pyproject.toml").exists(), "The ZIP must contain intrinsic-dimension-transformers/"
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
print("Project ready:", PROJECT)


## 1. Install compatible dependencies

This installs the project and its model/test extras. Compatible packages already installed are retained, including Colab's CUDA PyTorch. No paid API or Hugging Face token is required for the default public model. If a preloaded package changes, restart the session and rerun from the first cell; the extracted project folder can be reused.

In [ ]:
before = {name: getattr(sys.modules[name], "__version__", None)
          for name in ["numpy", "scipy", "matplotlib", "torch", "transformers"] if name in sys.modules}
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[model,test]"], check=True)
    changed = [name for name, version in before.items()
               if version is not None and version != importlib.metadata.version(name)]
    if changed:
        raise RuntimeError("Installed versions changed for preloaded packages: " + str(changed)
                           + ". Use Runtime > Restart session, then rerun cells.")
else:
    print("Local validation: dependency installation skipped")
from intrinsic_dim.io import versions
print(versions())


## 2. Optional persistent Google Drive storage

Drive storage was requested for this corrected run. Run this cell and authorize the Drive prompt to preserve checkpoints and activations. Set `SAVE_TO_DRIVE=False` in the first cell only if you choose temporary runtime storage instead. Model weights stay in the runtime cache; no weights are included in exported results.

In [ ]:
if SAVE_TO_DRIVE:
    assert IN_COLAB, "Google Drive mounting is a Colab-only option"
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT = Path("/content/drive/MyDrive/intrinsic_dimension/qwen_v2_fp32")
else:
    OUTPUT = PROJECT / "artifacts/qwen_v2_fp32"
CACHE = Path("/content/model_cache") if IN_COLAB else PROJECT / ".cache/huggingface"
print("Real-model results and checkpoints:", OUTPUT)
print("Model download cache:", CACHE)


## 3. Confirm the GPU and configure the experiment

For the real experiment the notebook requires CUDA, rather than silently running on CPU. If no GPU is available, change the runtime or try another session. Hardware allocation varies. All 29 standard hidden-state tuple entries are selected: embedding output plus 28 blocks; the final Qwen output includes RMSNorm. Automatic dtype uses native bfloat16 on supported GPUs and float32 on T4/older GPUs, avoiding a forced float16 exponent range. Batch size 1 is the starting point. Full Qwen GPU execution still requires validation in your session.

In [ ]:
if RUN_REAL_MODEL:
    import torch
    assert torch.cuda.is_available(), "No CUDA GPU. Select Runtime > Change runtime type > GPU, then reconnect."
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory free/total: {free / 2**30:.2f}/{total / 2**30:.2f} GiB")
else:
    print("Local notebook validation: real-model cells are disabled")

from intrinsic_dim.config import load_config
from intrinsic_dim.cli import main
import yaml
config = load_config("configs/default.yaml", device="cuda" if RUN_REAL_MODEL else "cpu",
                     batch_size=BATCH_SIZE, output_dir=str(OUTPUT), cache_dir=str(CACHE),
                     revision=MODEL_REVISION)
config.dtype = "float32"          # Explicit and consistent with the successful pilot.
config.figure_dpi = FIGURE_DPI
config.layers = None              # All available layers.
config.validate()
RUN_CONFIG = PROJECT / "artifacts/colab_run.yaml"
RUN_CONFIG.parent.mkdir(parents=True, exist_ok=True)
RUN_CONFIG.write_text(yaml.safe_dump(config.to_dict(), sort_keys=False))
print("Model:", config.model_id, "Preprocessing:", config.preprocessing, "PNG DPI:", config.figure_dpi)


## 4. Tests and synthetic end-to-end validation

These checks do not download Qwen. All mathematical tests must pass. Synthetic figures are labeled and stored in a different folder; they cannot answer the research question. Their output also confirms the high-resolution rendering pipeline works.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "--disable-warnings"], check=True)
main(["smoke-test", "--config", "configs/smoke_test.yaml"])
print("Validation passed; synthetic output is not a Qwen result.")


## 5. Audit the shipped dataset

Version 2 has 640 prompts, 160 per task, with 20 named families per category. Ten families (80 prompts) per category are held out. Source-linked factual fixtures span multiple domains; coding remains Python output prediction. Family labels do not prove independence. This cell checks actual Qwen token lengths and ensures at least 32 length-matched examples per category and eight retained factual/reasoning families. It stops before weights load if the design fails. Human ambiguity review is still pending. No arbitrary task-label permutation p-values are produced.

In [ ]:
from intrinsic_dim.dataset import load_dataset, audit
from intrinsic_dim.preflight import prepare_dataset, preflight
if IN_COLAB:
    rows, design_check = prepare_dataset(config)  # Tokenizer only; no model weights.
else:
    rows = load_dataset(config.dataset_path)
    design_check = preflight(rows, config)  # Previously verified exact Qwen counts.
assert design_check["status"] == "passed", design_check["errors"]
print(design_check["counts"])
print("Length matching:", design_check["length_matching"])
main(["audit", "--config", str(RUN_CONFIG)])
print(audit(rows)["categories"])
print("Selected primary split:", config.split)


## 6. Download Qwen and extract activations

Run this cell for the real model. It downloads public Qwen2.5-1.5B weights, uses inference mode, and selects the final **non-padding input token** before any generated answer. It saves one float32 NPY array per layer plus metadata.

**If memory runs out:** set `BATCH_SIZE=1` in the first cell, rerun configuration and this extraction cell with the same output directory. Compatible completed batches resume. Changing model revision/dataset/device/dtype/layers/seed requires a fresh output directory. If Drive was not enabled, a deleted runtime cannot preserve checkpoints.

In [ ]:
if RUN_REAL_MODEL:
    from intrinsic_dim.activations import extract
    extract(config)
    from intrinsic_dim.io import read_json
    metadata = read_json(OUTPUT / "activations/metadata.json")
    assert metadata["source"] == "model" and metadata["complete"]
    print("Extraction complete:", metadata["completed_rows"], "prompts")
    print("Actual device/dtype:", metadata["model"]["device"], metadata["model"]["dtype"])
    print("Resolved model revision:", metadata["model"]["resolved_revision"])
else:
    print("Local validation: Qwen download/extraction skipped")


## 7. Analyze every layer and generate HD figures

Analysis streams one layer at a time, applies separate raw/unit/standardized modes, computes the full available covariance spectra and neighborhood estimates, and runs sample-size/template/length controls. CPU analysis can take time after GPU extraction; a low GPU utilization message during this stage is expected. Primary raw-mode confidence intervals use 500 family-bootstrap replicates when valid. Unit/standardized modes and repeated subsamples are descriptive controls. Permutation p-values remain intentionally unperformed without an exchangeability design. Constant-token embeddings are labeled noninformative rather than treated as an extraction failure. Figures export as 600-DPI PNG plus vector PDF/SVG.

In [ ]:
if RUN_REAL_MODEL:
    main(["analyze", "--config", str(RUN_CONFIG)])
    REPORT_CONFIG = str(RUN_CONFIG)
else:
    REPORT_CONFIG = "configs/smoke_test.yaml"
main(["report", "--config", REPORT_CONFIG])
report_config = load_config(REPORT_CONFIG)
RESULTS = Path(report_config.output_dir) / "results"
FIGURES = Path(report_config.output_dir) / "figures"
from IPython.display import display, Markdown, Image
display(Markdown((RESULTS / "research_report.md").read_text()))
for name in ["02_layerwise_evolution", "03_pca_spectrum", "06_length_template_controls"]:
    display(Image(filename=str(FIGURES / (name + ".png")), width=1100))
print("Notebook previews are scaled; downloaded PNG/PDF/SVG files retain full resolution.")
print("Figures:", FIGURES)


## 8. Download the complete results ZIP

The ZIP includes code, configurations, documentation, sample metadata, reports, CSVs, source data and all high-resolution figures. Model weights are excluded. Activation arrays are also excluded unless you explicitly select `INCLUDE_ACTIVATIONS=True`; if you want to rerun analysis later, preserve those arrays in Drive or include them here. Use PDF/SVG for maximum sharpness in a paper.

When running in Colab this cell starts the ZIP download. If Drive was selected it also saves a persistent copy. Do not confuse the separately stored synthetic validation figures with real Qwen results.

In [ ]:
INCLUDE_ACTIVATIONS = False
EXPORT = Path("/content/intrinsic_dimension_v2_results.zip") if IN_COLAB else PROJECT / "intrinsic_dimension_v2_results.zip"
arguments = ["export", "--config", REPORT_CONFIG, "--destination", str(EXPORT)]
if INCLUDE_ACTIVATIONS:
    arguments.append("--include-activations")
main(arguments)
with zipfile.ZipFile(EXPORT) as archive:
    assert archive.testzip() is None
print(f"Results ZIP: {EXPORT} ({EXPORT.stat().st_size / 2**20:.1f} MiB)")
if SAVE_TO_DRIVE:
    import shutil
    saved = OUTPUT.parent / "intrinsic_dimension_v2_results.zip"
    shutil.copy2(EXPORT, saved)
    print("Also saved in Drive:", saved)
if IN_COLAB:
    files.download(str(EXPORT))
